In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.7 Checkpointing and resume

> 🎯 **Goal:** Save a complete training snapshot, model, optimizer, and step, then reload it into fresh objects and resume exactly where you left off.

**Why this matters.** Real training runs take hours or days and get interrupted: a machine reboots, a job is preempted, a power blip hits. Without checkpoints, an interruption means starting from scratch and losing all the progress. With them, you pick up exactly where you stopped. This lesson also produces the trained PocketLM that the rest of the course loads, so checkpointing is the bridge from this unit to everything after it.

**The intuition.** Think of a save point in a long video game. A good save records everything you need to continue, not just your character's position but your inventory, your quest progress, the whole state. Saving only the model weights would be like saving your character's location but forgetting your inventory: you would respawn standing in the right place but stripped of your gear. A complete checkpoint preserves the full game state so resuming is seamless, not a fresh start.

**The mechanics.** A useful checkpoint stores three things. First, the model's weights (what it has learned). Second, the optimizer's state, AdamW's running first and second moments from lesson 2, because those moments took many steps to build up and throwing them away would stall training on resume. Third, the step number, so the learning-rate schedule and logging continue from the right place. `save_checkpoint(path, model, opt, step=30)` writes all three to disk. To resume, you create fresh model and optimizer objects and call `load_checkpoint(path, fresh, fresh_opt)`, which pours the saved state into them and returns the step. The fresh model's weights should then match the original byte for byte.

In [ ]:
import tempfile
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, save_checkpoint, load_checkpoint)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(30):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
ckpt = os.path.join(tempfile.gettempdir(), "pocketlm_u5.pt")
save_checkpoint(ckpt, model, opt, step=30)
print("saved checkpoint at step 30")

**The resume step.** Now we build a brand-new model and a brand-new optimizer (as if we just relaunched the program on a fresh machine), then call `load_checkpoint` to pour the saved state into them. The function restores the weights and the optimizer moments and hands back the step number, so the next training loop could continue without missing a beat. The verification below checks two things: that the reported step is the 30 we saved, and that every parameter in the resumed model exactly equals the original.

In [ ]:
fresh = PocketLM(cfg)
fresh_opt = torch.optim.AdamW(fresh.parameters(), lr=1e-3)
resumed_step = load_checkpoint(ckpt, fresh, fresh_opt)
match = all(torch.equal(a, b) for a, b in zip(model.parameters(), fresh.parameters()))
print("resumed at step:", resumed_step, " weights match:", match)
assert resumed_step == 30
assert match

**What just happened.** The printout reports the resumed step (30) and that the weights match. The two assertions then make it official: the snapshot round-tripped perfectly, fresh objects in, identical model out. You can now interrupt and resume training at will, and you have produced the trained PocketLM that later units load from disk. That wraps up unit u5: you can now initialize a model sanely, optimize it with AdamW, schedule its learning rate, split data honestly, read its train-val curves, score it with perplexity, and save and resume it.

⚠️ **Common pitfalls**
- Saving only `model.state_dict()` and dropping the optimizer state: on resume, AdamW's moments reset to cold and training stutters or briefly destabilizes, exactly the warmup problem from lesson 3, all over again.
- Forgetting to save the step number: the learning-rate schedule and logging restart from zero, so the resumed run no longer follows the intended curve.
- Loading a checkpoint into a model built with a different config (different `n_layer`, `n_embd`, or `vocab_size`): the shapes will not line up and the load fails or silently corrupts.

🏋️ **Try it yourself.** After resuming, run a few more training steps on `fresh`, then save a second checkpoint at step 35 and reload that. Confirm the step advances correctly. As a contrast, try saving only the model (skip passing `opt`) and observe how the resumed optimizer starts cold, the concrete reason the full snapshot matters.